# 4 · Map the next observation with the same input recipe

The forest model works, but its weights do not say which four bands it saw, whether reflectance was decoded, or how missing pixels were filled. Copying the inference code into another experiment makes those choices easy to change accidentally.

We will keep those choices with the trained model, prove that they reproduce its existing predictions, and use the saved recipe on another forest observation.

**Before starting:** finish notebooks [2](02_sources.ipynb) and [3](03_inputs.ipynb), including notebook 3's **Keep this trained model** cell. Start a fresh kernel with `examples/` as the working directory. Use the repository environment (`uv sync`). This lesson uses the saved data and weights; it does not download imagery or train again.

**At the end:** `data/landcover/model_spec.yaml` records the input recipe, and `data/landcover/review/` contains an image crop and a dated land-cover TIFF to review together. Rerunning replaces these tutorial files.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
import yaml

import geosave as gs
from geosave.model import build_model
from geosave.model.spec import ModelSpec

folder = Path("data/landcover")
recipe = yaml.safe_load((folder / "model.yaml").read_text())
classes = yaml.safe_load((folder / "classes.yaml").read_text())

model = build_model(recipe)
weights = torch.load(folder / "weights.pt", map_location="cpu", weights_only=True)
model.load_state_dict(weights)
model.eval()

## Revisit one validation window

Use the same acquired validation split as notebook 3. We need just one 128 × 128 window to compare old and new preparation. Its image and label remain on the same grid. We are checking that a refactor preserves behavior, not measuring new model accuracy.

In [ ]:
image_path = sorted(Path("data/my_dataset/val/sentinel-2-l2a").glob("*.tif"))[0]
label_path = Path("data/my_dataset/val/label") / image_path.name

with gs.open_raster(image_path) as source:
    image = source.isel(y=slice(0, 128), x=slice(0, 128))
    image = image.load()

with gs.open_raster(label_path) as source:
    label = source["label"].isel(y=slice(0, 128), x=slice(0, 128))
    label = label.load()

image.gs.mask_and_scale().gs.plot(title="Validation image") + label.gs.plot(
    title="Reference labels"
)

## Start with the preparation we already used

Notebook 3 selected blue, green, red, and near-infrared in that order, decoded reflectance once, and filled missing reflectance with zero before making tensors. Keep the validity mask separately: filling an input must not turn a missing pixel into a reported land-cover class.

In [ ]:
bands = ["B02", "B03", "B04", "B08"]
reflectance = image[bands].gs.mask_and_scale()
valid = reflectance.to_array().notnull().all("variable")

prepared = reflectance.fillna(0)
manual = prepared.gs.to_tensor(dtype="float32")

with torch.inference_mode():
    batch = manual.unsqueeze(0)
    before = model(image=batch)[0]

## Write down those choices

`model.yaml` describes the neural network. `model_spec.yaml` below describes the values we feed it and how its output returns to a map. `!ref` refers to an earlier named value; it does not open a file or select an observation.

There is deliberately no new normalization: these weights were trained on decoded reflectance. Adding a pretrained encoder's normalization now would change their input distribution.

This uses the current ModelSpec v3 API. It executes the installed calls named in a recipe, so use recipes you have reviewed.

In [ ]:
spec_path = folder / "model_spec.yaml"
spec_path.write_text("""schema_version: 3

rasters:
  optical:
    variables: [B02, B03, B04, B08]
    dims: [y, x]
    require_crs: true

preprocessing:
  reflectance:
    call: !ref optical.gs.mask_and_scale
  image:
    call: !ref reflectance.fillna
    kwargs:
      value: 0

encoding:
  pixels:
    call: geosave.model.processing.to_tensor
    kwargs:
      raster: !ref image
      dims: [band, y, x]
      dtype: float32

inputs:
  image: !ref pixels

pixels: [image]

tensors:
  image:
    shape: [4, 128, 128]
    dtype: float32

decoding:
  scores:
    call: geosave.model.processing.to_raster
    kwargs:
      value: !ref prediction
      reference: !ref optical
      dims: [band, y, x]
      name: scores
""")

spec = ModelSpec.load(spec_path)
spec

## Check that the recipe preserves the model's inputs

Give the recipe the bands in reverse file order to make the purpose visible: it selects the declared order before conversion. The table compares what the model receives. Pixel reads happen in `encode`; preprocessing still works with native xarray values.

In [ ]:
reordered = image[["B08", "B04", "B03", "B02"]]
prepared = spec.preprocess({"optical": reordered})

values = spec.encode(prepared)
inputs = spec.model_inputs(values)

difference = (manual - inputs["image"]).abs()
maximum = difference.amax(dim=(1, 2))

pd.DataFrame(
    {
        "band": bands,
        "manual_mean": manual.mean(dim=(1, 2)).numpy(),
        "recipe_mean": inputs["image"].mean(dim=(1, 2)).numpy(),
        "largest_difference": maximum.numpy(),
    }
)

## Check the prediction, then put it back on the map

Identical inputs should produce identical class scores in evaluation mode. Decode those scores on the original grid, choose class IDs, and restore the legend. We keep this small numerical check because it protects the trained model when changing its preparation code.

In [ ]:
with torch.inference_mode():
    batch = inputs["image"].unsqueeze(0)
    after = model(image=batch)[0]

torch.testing.assert_close(inputs["image"], manual)
torch.testing.assert_close(after, before)

scores = spec.decode(after, values)["scores"]
scores = scores.assign_coords(band=sorted(classes["class_map"]))
cover = scores.idxmax("band")
cover = cover.where(valid, classes["ignore_index"])
cover = cover.astype("uint8")

cover = cover.rename("label")
cover = cover.gs.write_nodata(classes["ignore_index"])
cover = cover.gs.write_legend(classes["class_map"], classes["color_map"])

label.gs.plot(title="Reference labels") + cover.gs.plot(
    title="Same trained model, saved recipe"
)

## Reopen the saved recipe for a forest observation

Now leave the labeled validation tile and use the most recent scene in the forest catalog. This is an application area, not another accuracy test. Select a central 128 × 128 window: it has the spatial size expected by this trained model. The crop is a review window, not a map of the whole forest.

The model, band selection, scaling, and missing-pixel policy stay fixed. For a full forest map, apply the same preparation before the tiling and score-merging workflow in notebook 3.

In [ ]:
spec = ModelSpec.load(spec_path)
items = gs.open_vector("data/khdtk/catalog.parquet")
items = items.sort_values("datetime")

tree = items.iloc[[-1]].gs.to_datatree()
scene = tree["sentinel-2-l2a"].gs.to_dataset()
if "time" in scene.dims:
    scene = scene.squeeze("time")

y = (scene.sizes["y"] - 128) // 2
x = (scene.sizes["x"] - 128) // 2
observation = scene.isel(y=slice(y, y + 128), x=slice(x, x + 128))
observation = observation.load()
tree.close()

observation.gs.mask_and_scale().gs.plot(title="Forest observation to review")

In [ ]:
prepared = spec.preprocess({"optical": observation})
values = spec.encode(prepared)
inputs = spec.model_inputs(values)

with torch.inference_mode():
    batch = inputs["image"].unsqueeze(0)
    prediction = model(image=batch)[0]

scores = spec.decode(prediction, values)["scores"]
scores = scores.assign_coords(band=sorted(classes["class_map"]))
valid = prepared["reflectance"].to_array().notnull().all("variable")

forest_cover = scores.idxmax("band")
forest_cover = forest_cover.where(valid, classes["ignore_index"])
forest_cover = forest_cover.astype("uint8")

forest_cover = forest_cover.rename("label")
forest_cover = forest_cover.assign_coords(time=observation.time)
forest_cover = forest_cover.gs.write_nodata(classes["ignore_index"])
forest_cover = forest_cover.gs.write_legend(classes["class_map"], classes["color_map"])

forest_cover.gs.plot(title="Land-cover prediction for review")

## Keep the image and prediction together

A reviewer needs to see the source image beside the model's claim. Save both with their acquisition date so the files remain meaningful outside this notebook. The recipe and trained weights stay together in their parent folder.

In [ ]:
review = folder / "review"
observation.gs.to_cog(review, split_asset=(), format_paths="image-{time:%Y%m%d}.tif", overwrite=True)
paths = forest_cover.gs.to_cog(
    review, format_paths="land-cover-{time:%Y%m%d}.tif", overwrite=True
)

with gs.open_raster(paths[0]) as saved:
    display(saved.gs.plot(title="Reopened review map"))

## More things you can do with this recipe (optional)

The saved recipe and review maps complete this lesson. These are possible next steps for your project.

### Keep a small reference case with every model version
Retain the validation window and expected scores. Check them before mapping a new season or after upgrading dependencies. That can catch an accidental band reorder or second reflectance scaling before it reaches a whole region.

### Reuse the recipe when preparing a new training set
Apply the saved preprocessing before constructing the training Dataset, as in notebook 3. Keep label preparation separate: class IDs must not receive reflectance scaling. Keep paired augmentation in the training loop. Compare the prepared image values before starting a new experiment.

### Plan a model your colleague can reopen
You now have trained weights, the architecture, class definitions, and an input recipe. The planned release lesson will package them, reload them in a separate process, and make a map without importing the training notebook.

### Test a pretrained encoder fairly
A pretrained model may require different bands or normalization. Make that a new experiment with its own recipe and the same geographic splits. This is the next planned lesson, not a reason to silently modify this model's inputs.